# Revision Overview: What Has Been Done and What It Shows

This notebook reviews the work on the revision of "From Graph k-Coloring to Grover's Circuits via Logic Synthesis" in
one place. It follows the argument of the revised paper:

1. **The problem.** Graph coloring as a Grover search, its encoding and the three ways of preparing the search space.
2. **Three errors in the original evaluation**, each shown on a small example: the diffusion operator, the vertex
   labels of the baseline circuits, and the counting of ancilla qubits.
3. **The explicit cost model** that replaces the manuscript's estimates: decompositions of multi-controlled gates,
   exact Clifford+T gadgets, and measurement-based uncomputation.
4. **The corrected comparison** of every synthesis method with every baseline on the paper's nine graphs.
5. **Why our oracle ties the Belletti-Simple baseline**, and where the width claims of the manuscript came from.
6. **Optimizing the synthesized oracle**: fewer AND gates, balanced logic, Caterpillar's mapping strategies, the shape
   of the final conjunction with gate scheduling, the T-depth of every method before and after optimization, and
   measurement-based uncomputation priced fairly for every method.
7. **Cheaper preparation of the valid colors**: exact repeat-until-success rotations, Hamming-weight phasing and
   amplitude amplification, priced on the complete search.
8. **Key results**, computed from the sections above, and the work that remains.
9. **The Pareto view**: which approach and setting is best for each metric, and the trade-offs between qubits, T-count
   and T-depth.
10. **Two-qubit gates and routing** (optional material: whether the paper covers current hardware is decided later),
    on FakeTorino and on a square grid that grows with the circuit.

The Saha–Belletti constructions and the synthesis methods are treated as *approaches* to building the same Grover
circuit, hand-constructed or synthesized, under shared settings (decomposition of multi-controlled gates, conjunction
shape and scheduling, measurement-based uncomputation, preparation, pricing); each synthesized approach is compared
with the best hand-constructed approach for each metric. The code and some tables still call the latter "baselines".

Every number is computed in this run, from the saved nine-graph study named by `LOGIQUE_STUDY_RUN`, from the later
studies found by name under `dump/` (repriced, scheduled, balanced, tree shapes, best-fit depths, grid routing), or
from circuits built here. Figures are saved as PNG and SVG files next to the executed notebook.

## Setup

In [ ]:
from dataclasses import replace
from pathlib import Path
import json
import math
import os

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import SVG, Image, Markdown, display
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from matplotlib.ticker import LogLocator
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.quantum_info import Operator, Statevector

from logique import encode_coloring, read_dimacs
from logique.benchmarks import analysis, costs
from logique.benchmarks import preparations as prepared
from logique.benchmarks.study import expand_oracle
from logique.circuits import preparation as prep
from logique.circuits.search import baseline_instance, logique_instance
from logique.paths import native_executable, new_dump_directory
from logique.synthesis import synthesize

ROOT = Path(os.environ.get("LOGIQUE_WORKSPACE", Path.cwd())).resolve()
if "LOGIQUE_STUDY_RUN" not in os.environ:
    raise RuntimeError("Set LOGIQUE_STUDY_RUN to the folder of the completed nine-graph study")
RUN = (ROOT / os.environ["LOGIQUE_STUDY_RUN"]).resolve()
if "LOGIQUE_OUTPUT" in os.environ:
    OUT = Path(os.environ["LOGIQUE_OUTPUT"]).resolve()
else:
    OUT = new_dump_directory("revision-overview", root=ROOT / "dump")
FIGURES = OUT / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
EXECUTABLE = native_executable(
    os.environ.get("LOGIQUE_NATIVE_EXECUTABLE")
    or ROOT / "external/native/boolean_synthesis/build/boolean_synthesis"
)
GRAPH_DIR = ROOT / "datasets/graphs/benchmarks"

rows, manifest = analysis.load_study(RUN)
settings = manifest["settings"]
GRAPHS = [Path(p).stem for p in settings["inputs"]]
LOGIQUE = [m for m in settings["methods"] if not m.startswith("sb_")]
BASELINES = [m for m in settings["methods"] if m.startswith("sb_")]
POLICIES = settings["mcx_policies"]
REGIMES = settings["regimes"]
problems = {g: replace(read_dimacs(GRAPH_DIR / f"{g}.col"), precolored={}) for g in GRAPHS}

SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
INK, SECONDARY, MUTED, GRID, AXIS = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
plt.rcParams.update(
    {
        "figure.dpi": 110,
        "font.size": 9,
        "axes.edgecolor": AXIS,
        "axes.labelcolor": SECONDARY,
        "xtick.color": SECONDARY,
        "ytick.color": SECONDARY,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.color": GRID,
        "grid.linewidth": 0.6,
        "axes.axisbelow": True,
    }
)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
KEY = {}  # numbers quoted in the summary of Section 8


def show_figure(figure, stem: str) -> None:
    """Save a Matplotlib figure as PNG and SVG, then display it."""
    figure.savefig(FIGURES / f"{stem}.svg", bbox_inches="tight")
    figure.savefig(FIGURES / f"{stem}.png", dpi=140, bbox_inches="tight")
    display(Image(filename=str(FIGURES / f"{stem}.png")))
    plt.close(figure)


def show_svg(svg: bytes, stem: str) -> None:
    (FIGURES / f"{stem}.svg").write_bytes(svg)
    display(SVG(data=svg))


def log_axis(axis):
    axis.set_major_locator(LogLocator(subs=(1.0, 2.0, 5.0)))
    axis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:g}"))
    axis.set_minor_formatter(plt.NullFormatter())


def role_groups(instance, circuit):
    """Wire groups for drawing: one register per vertex, then each role."""
    groups = [
        (f"v{vertex}", [instance.data[i] for i in inputs])
        for vertex, inputs in instance.encoding.input_mapping.items()
    ]
    role = "ancillas" if instance.method.startswith("sb_") else "workspace"
    groups.append(("e" if role == "ancillas" else "w", instance.roles[role]))
    groups.append(("a", list(range(instance.oracle.num_qubits, circuit.num_qubits))))
    groups.append(("out", instance.roles["output"]))
    return [(name, wires) for name, wires in groups if wires]


def labelled(circuit, groups):
    """Copy a circuit onto named registers so that drawings show qubit roles."""
    registers = [QuantumRegister(len(wires), name) for name, wires in groups]
    drawn = QuantumCircuit(*registers, global_phase=circuit.global_phase)
    position = {}
    for register, (_, wires) in zip(registers, groups):
        position.update(zip(wires, register))
    for inst in circuit.data:
        drawn.append(inst.operation, [position[circuit.find_bit(q).index] for q in inst.qubits])
    return drawn


def expanded(instance, policy="throwaway_tree"):
    """Expand an oracle as the study does: MCX policy, then relative-phase pairs."""
    circuit, _ = expand_oracle(instance, policy)
    return costs.paired_relative_phase_toffolis(circuit)


ft = rows[rows.metric_level == "fault-tolerant"]
search = analysis.search_costs(rows)
print(f"Study {manifest['run_id']}: {len(manifest['cases'])} cases, {len(GRAPHS)} graphs, "
      f"{len(LOGIQUE)} synthesis methods, {len(BASELINES)} baselines, policies {', '.join(POLICIES)}")
print("Artifacts:", OUT)

**Repriced measurement-based column.** The detector that pairs each AND with its uncomputation for measurement-based
pricing required the AND's target to start in |0⟩ and to see no other write before the uncomputation. The baselines
set their flags to |1⟩ and wrap open controls in X gates, and Bennett-style mappings flip targets in between, so many
of their temporary ANDs were priced as unitary although measurement can uncompute them (corrected on 15 November;
Section 6 shows the effect on K₃). The nine-graph study is being repriced without routing; once that run is complete,
its measurement-based rows replace the study's here, so every section below uses the corrected pricing. The unitary
and routed rows are unchanged.

In [ ]:
import glob

MEASURED = "fault-tolerant, measurement-assisted"


def latest_run(description):
    """Newest complete study run with this description, or None."""
    for path in sorted(glob.glob(str(ROOT / "dump" / f"*-{description}")), reverse=True):
        record = json.loads((Path(path) / "run.json").read_text())
        if record.get("status") == "complete":
            return Path(path)
    return None


REPRICED = latest_run("repriced-comparison")
if REPRICED is not None:
    repriced = analysis.load_study(REPRICED)[0]
    rows = pd.concat([rows[rows.metric_level != MEASURED], repriced[repriced.metric_level == MEASURED]], ignore_index=True)
    ft = rows[rows.metric_level == "fault-tolerant"]
    search = analysis.search_costs(rows)
    display(Markdown(f"Measurement-based rows taken from the repriced run `{REPRICED.name}`."))
else:
    display(Markdown("**The repriced run is not complete yet: the measurement-based rows below are the study's, "
                     "which understate the savings of every method except the XAG flow.**"))

## 1. The problem

A graph with vertices V and edges E is k-colorable if each vertex can receive one of k colors so that adjacent
vertices differ. Each vertex holds its color in c = ⌈log₂ k⌉ qubits, least-significant bit first, so when k is not a
power of two some of the 2^c codes are unused. Grover's algorithm repeats an **oracle**, which marks proper
colorings, and a **diffusion operator**, which reflects about the starting state. Three settings are compared
throughout:

| Setting | Starting state | What the oracle checks |
|---|---|---|
| **R1** | all 2^c codes per vertex (Hadamard gates) | edges and the unused codes |
| **R2** | only the k valid codes per vertex | edges and the unused codes |
| **R3** | only the k valid codes per vertex | edges only (the unused codes never occur) |

Starting only in valid codes shrinks the search space from 2^(c·|V|) to k^|V| states, so fewer iterations are needed;
the price is a rotation per vertex in every preparation and inverse preparation. The table gives the nine graphs of
the paper, their search spaces and the optimal number of iterations.

In [ ]:
from logique.visualization.graphs import draw_colored_graph

problem = problems["K3"]
graph = problem.to_networkx()
draw_colored_graph(graph, name=str(FIGURES / "k3.png"), pos=nx.circular_layout(graph), figsize=(3.2, 2.8), node_size=1200)
plt.gcf().axes[0].set_title("K3 with three colors")
show_figure(plt.gcf(), "k3")

domains = (
    rows[rows.component == "search"]
    .drop_duplicates(["benchmark", "regime"])
    .pivot(index="benchmark", columns="regime", values=["domain", "iterations"])
)
overview = pd.DataFrame(
    {
        "vertices": {g: len(p.vertices) for g, p in problems.items()},
        "edges": {g: len(p.edges) for g, p in problems.items()},
        "colors k": {g: p.colors for g, p in problems.items()},
        "bits per vertex": {g: max(1, (p.colors - 1).bit_length()) for g, p in problems.items()},
        "proper colorings": rows.drop_duplicates("benchmark").set_index("benchmark").solutions.astype("Int64"),
    }
).reindex(GRAPHS)
for regime in ("R1", "R2"):
    overview[f"search space {regime}"] = domains[("domain", regime)].map(lambda v: f"{float(v):.2e}" if v == v else "")
    overview[f"iterations {regime}"] = domains[("iterations", regime)].astype("Int64")
display(overview)
display(Markdown("myciel5 has too many colorings to count, so its searches are compared per iteration."))

The oracle is synthesized from a Boolean description of the coloring constraints. Logique writes the predicate as an
Xor-And-Inverter Graph (XAG): every AND node becomes a Toffoli-type gate, while XOR and NOT nodes are free (CNOT and X
gates). The two XAGs below are for K₃: the edge-only predicate (R3) and the complete predicate with the checks of the
unused code (R1 and R2).

In [ ]:
from logique.visualization.boolean import signal_names

def boolean_expression(network, input_labels=None):
    """Write out every gate in full, giving one expression per output."""
    names = signal_names(network, input_labels=input_labels)
    nodes = {node["id"]: node for node in network["nodes"]}
    symbol = {"and": " ∧ ", "xor": " ⊕ "}
    memo = {}

    def term(node_id):
        if node_id not in memo:
            node = nodes.get(node_id, {"kind": "input"})
            if node["kind"] in ("input", "constant"):
                memo[node_id] = names[node_id]
            else:
                memo[node_id] = "(" + symbol[node["kind"]].join(
                    literal(f["node"], f["inverted"]) for f in node["fanins"]) + ")"
        return memo[node_id]

    def literal(node_id, inverted):
        node = nodes.get(node_id, {})
        fanins = node.get("fanins", [])
        if inverted and node.get("kind") == "and" and all(f["inverted"] for f in fanins):
            return "(" + " ∨ ".join(term(f["node"]) for f in fanins) + ")"
        return f"¬{term(node_id)}" if inverted else term(node_id)

    return [literal(out["node"], out["inverted"]) for out in network["outputs"]]

In [ ]:
from logique.visualization.boolean import graphviz_graph

labels = [None] * encode_coloring(problem).input_count
for vertex, inputs in encode_coloring(problem).input_mapping.items():
    for bit, i in enumerate(inputs):
        labels[i] = f"x_{vertex},{bit}"
for title, encoding, stem in (
    ("Edge-only predicate (R3)", encode_coloring(replace(problem, colors=4)), "xag-edges"),
    ("Complete predicate (R1, R2)", encode_coloring(problem), "xag-complete"),
):
    network = synthesize(encoding, "xag", executable=EXECUTABLE).metadata["network"]
    show_svg(graphviz_graph(network, title, input_labels=labels).pipe(format="svg"), stem)
    # print(encoding.verilog())
    for y, text in enumerate(boolean_expression(network, labels)):
        print(f"{title}: y{y} = {text}")

## 2. Three errors in the original evaluation

**The diffusion operator.** The manuscript prepares only the valid colors but its benchmark generator reflected about
the Hadamard state, which covers every code. On K₃ with three colors this leaks probability into the unused code and
roughly halves the success probability after one iteration. Reflecting about the actual preparation fixes it.

In [ ]:
from logique.circuits.grover import kickback_grover_circuit

edge_only = encode_coloring(replace(problem, colors=4))
oracle = synthesize(edge_only, "xag", executable=EXECUTABLE)
inputs = oracle.metadata["input_qubits"]
vertex_preparation = prep.shukla_vedula(3)
preparation = QuantumCircuit(len(inputs))
for start in range(0, len(inputs), 2):
    preparation.compose(vertex_preparation, [start, start + 1], inplace=True)
hadamard = QuantumCircuit(len(inputs))
hadamard.h(range(len(inputs)))
matching = kickback_grover_circuit(oracle, preparation=preparation, iterations=1)
historical = kickback_grover_circuit(oracle, preparation=preparation, reflection_preparation=hadamard, iterations=1)

assignments = [edge_only.decode(b) for b in range(1 << len(inputs))]
unused = np.array([any(c >= 3 for c in a.values()) for a in assignments])
proper = np.array([problem.is_valid(a) for a in assignments])
groups = {"proper coloring": proper, "valid codes, conflict": ~unused & ~proper, "uses the unused code": unused}
reflections = pd.DataFrame(
    {
        "before the iteration": {k: float(Statevector(preparation).probabilities()[m].sum()) for k, m in groups.items()},
        "reflection about the preparation": {k: float(Statevector(matching).probabilities(inputs)[m].sum()) for k, m in groups.items()},
        "historical Hadamard reflection": {k: float(Statevector(historical).probabilities(inputs)[m].sum()) for k, m in groups.items()},
    }
)
display(reflections.round(3))
figure, ax = plt.subplots(figsize=(7.5, 3))
reflections.plot.bar(ax=ax, rot=0, color=[SERIES[3], SERIES[0], SERIES[1]], width=0.75, edgecolor="white")
ax.set_ylabel("Probability")
ax.set_xlabel("")
ax.grid(axis="x", visible=False)
ax.legend(frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.12))
show_figure(figure, "reflection")
KEY["k3 success"] = reflections.loc["proper coloring"].iloc[1:].to_dict()

**Vertex labels of the baselines.** The baseline package uses each vertex label as a register index. The historical
replay passed labels starting at 1, so the Saha et al. variant skipped every edge of the last vertex. On a
three-vertex path with two colors, one iteration should find one of the two proper colorings with certainty; with the
historical labels it does no better than guessing.

In [ ]:
from contextlib import redirect_stdout
import io

from logique.synthesis.saha_belletti import VARIANTS, comparison_circuit

path = read_dimacs(ROOT / "datasets/graphs/execution/linear3.col")
alternating = (0b010, 0b101)
with redirect_stdout(io.StringIO()):  # the baseline package prints its iteration count
    labels_table = pd.DataFrame(
    {
        name: {
            variant: float(
                sum(
                    Statevector(comparison_circuit(path, variant, iterations=1, preserve_labels=keep)).probabilities([0, 1, 2])[b]
                    for b in alternating
                )
            )
            for variant in VARIANTS
        }
        for name, keep in (("historical labels", True), ("normalized labels", False))
    }
    )
display(labels_table.round(3))

**Counting ancilla qubits.** The manuscript counted the baselines' ancillas with a rule of its own: n − 2 ancillas
for every n-control gate of the Grover circuit except the last one, which is the gate of the diffusion operator,
whereas Logique's circuits were counted by the qubits they allocate. The correct width is the *peak* number of qubits
in use at once, which depends on how the multi-controlled gates are decomposed. The rule over-counts. The baseline
undoes its compute block in mirror order after the kick-back gate, and the rule charges ancillas to every gate and to
its mirror, although the mirror reuses them. Leaving out the diffusion gate costs nothing, because that gate reuses the
oracle's workspace, which is clean when it acts. On graphs with c = 2, the comparators and unused-code checks are plain
Toffoli gates, the only gate charged is the final conjunction, and the rule equals the peak under the throwaway tree.
On graphs with c ≥ 3, the comparators are multi-controlled and charged twice, and the rule exceeds the peak.

The table applies the rule to the baseline's full Grover circuit with the manuscript's labels, where it reproduces
every archived width, and the peak count to the Belletti-Simple oracle with its unused-code checks (the setting of the
manuscript's baselines), next to Logique's oracle. Under the historical rule the baseline looks wider than Logique's
oracle on every graph with c ≥ 3; counted at its peak with its best decomposition, it is narrower on every graph
except K₃.

The best decomposition is the narrowest of the five for each graph, so it changes from graph to graph: Iten et al.'s
dirty-ancilla construction (tied with Vale et al.'s) on the graphs with c = 2, and Vale et al.'s ancilla-free
construction on the others, where it costs 6 to 56 times the T gates of Khattar–Gidney plus thousands of arbitrary
rotations. The Khattar–Gidney column, with two clean ancillas, is the narrowest practical choice: it is within two
qubits of the minimum on every graph and the cheapest in T among the decompositions that restore their ancillas.

In [ ]:
ARCHIVED = pd.read_csv(ROOT / "experiments/manuscript/legacy/data/output/benchmark_results.csv").set_index("benchmark")


def historical_width(problem):
    """The manuscript's rule, applied as the manuscript did: to the baseline's full Grover circuit.

    Every gate with three or more controls is charged n - 2 ancillas, except the last one,
    which is the gate of the diffusion operator. Returns the circuit and the width.
    """
    with redirect_stdout(io.StringIO()):  # the baseline package prints its iteration count
        circuit = comparison_circuit(problem, "simple", iterations=1, preserve_labels=True)
    gates = [inst.operation for inst in circuit.data if inst.operation.name == "mcx"]
    return circuit, circuit.num_qubits + sum(max(0, gate.num_ctrl_qubits - 2) for gate in gates[:-1])


oracle_rows = ft[ft.component == "oracle"]
width_rows = []
for g in GRAPHS:
    _, summed = historical_width(problems[g])
    assert summed == ARCHIVED.loc[g, "sb_simple_width"], (g, summed)  # reproduces the manuscript
    simple = oracle_rows[(oracle_rows.benchmark == g) & (oracle_rows.method == "sb_simple") & (oracle_rows.regime == "R2")]
    xag = oracle_rows[(oracle_rows.benchmark == g) & (oracle_rows.method == "xag") & (oracle_rows.regime == "R3")]
    width_rows.append(
        {
            "graph": g,
            "Belletti-Simple, historical rule": summed,
            "Belletti-Simple, peak (throwaway tree)": int(simple[simple.mcx_policy == "throwaway_tree"].qubits.iloc[0]),
            "Belletti-Simple, peak (best decomposition)": int(simple.qubits.min()),
            "Belletti-Simple, peak (Khattar–Gidney)": int(simple[simple.mcx_policy == "2_clean_kg24"].qubits.iloc[0]),
            "Logique XAG (edge-only)": int(xag.qubits.iloc[0]),
        }
    )
widths = pd.DataFrame(width_rows).set_index("graph")
display(widths)
bits_per_color = {g: max(1, (problems[g].colors - 1).bit_length()) for g in GRAPHS}
excess = widths["Belletti-Simple, historical rule"] / widths["Belletti-Simple, peak (throwaway tree)"] - 1
two_bits = [g for g in GRAPHS if bits_per_color[g] == 2]
more_bits = [g for g in GRAPHS if bits_per_color[g] >= 3]
display(Markdown(
    f"The historical rule reproduces all {len(GRAPHS)} archived widths. It equals the peak under the throwaway tree "
    f"on the graphs with c = 2 ({', '.join(two_bits)}: excess {excess[two_bits].abs().max():.0%}) and exceeds it by "
    f"{excess[more_bits].min():.0%} to {excess[more_bits].max():.0%} on the graphs with c ≥ 3."
))
figure, ax = plt.subplots(figsize=(9, 3.4))
x = np.arange(len(widths))
for i, (column, color) in enumerate(zip(widths.columns, [SERIES[1], SERIES[3], SERIES[2], SERIES[4], SERIES[0]])):
    ax.bar(x + (i - 2) * 0.16, widths[column], width=0.16, color=color, edgecolor="white", label=column)
ax.set_yscale("log")
log_axis(ax.yaxis)
ax.set_xticks(x, widths.index, rotation=30, ha="right")
ax.set_ylabel("Qubits")
ax.grid(axis="x", visible=False)
ax.legend(frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.25))
show_figure(figure, "width-accounting")


**The width rule on K₃, drawn.** Belletti-Simple keeps one flag qubit per constraint (register e: three edges and
three unused-code checks) until a final gate combines them. The comparators work in place on the data register, and
each unused-code check is a single Toffoli gate, because three colors leave only the code 11 unused. The Grover circuit
of the manuscript therefore holds two gates with three or more controls: the conjunction of the six flags onto the
output, and the gate of the diffusion operator, which has five controls on the data register. The first drawing shows
that circuit (Hadamard preparation, oracle and the package's diffusion operator) before decomposition. The rule leaves
out the last gate, the diffusion gate, and charges the conjunction 6 − 2 = 4 ancillas: 13 + 4 = 17 qubits.

Counted at its peak, the conjunction needs the same four ancillas. It acts on the output without a mirror, so it
cannot leave garbage, and the throwaway-tree policy falls back to the clean tree, which restores them (register a in
the second drawing, which shows the oracle alone). The diffusion gate then reuses the flags and these ancillas, which
are clean when it acts, so the peak is also 17 qubits: on K₃ the rule and the peak agree. In the second drawing,
`Rccx` marks relative-phase Toffoli gates: the compute block's Toffoli gates may carry relative phases because their
mirror cancels them.

In [ ]:
k3 = problems["K3"]
k3_full, k3_rule = historical_width(k3)
k3_controls = [inst.operation.num_ctrl_qubits for inst in k3_full.data if inst.operation.name == "mcx"]
assert len(k3_controls) == 2  # the conjunction, then the diffusion gate
k3_bits = max(1, (k3.colors - 1).bit_length())
k3_groups = [(f"v{v}", [i * k3_bits + b for b in range(k3_bits)]) for i, v in enumerate(k3.vertices)]
k3_groups += [("e", list(range(len(k3.vertices) * k3_bits, k3_full.num_qubits - 1))), ("out", [k3_full.num_qubits - 1])]
display(Markdown(
    f"**Belletti-Simple on K3, historical rule**: {k3_full.num_qubits} qubits allocated + "
    f"({k3_controls[0]} − 2) for the conjunction = {k3_rule} qubits (archived: {ARCHIVED.loc['K3', 'sb_simple_width']}). "
    f"The last gate, the diffusion gate with {k3_controls[-1]} controls, is not charged."
))
show_figure(labelled(k3_full, k3_groups).draw("mpl", fold=45), "k3-simple-historical")

# Peak count under the throwaway-tree policy. The compute block holds only Toffoli
# gates, and the conjunction acts on the output without a mirror, so it must
# restore its ancillas: the policy falls back to the clean tree (controls - 2 ancillas).
k3_simple = baseline_instance(k3, "simple", "R2")
k3_peak = expanded(k3_simple, "throwaway_tree")
display(Markdown(
    f"**Belletti-Simple on K3, peak (throwaway tree)**: {k3_peak.num_qubits} qubits, "
    f"{costs.ft_counts(k3_peak)['t_count']} T gates. The conjunction is expanded with the clean tree on "
    f"{k3_peak.num_qubits - k3_simple.oracle.num_qubits} new ancillas (register a), which it restores."
))
show_figure(labelled(k3_peak, role_groups(k3_simple, k3_peak)).draw("mpl", fold=45), "k3-simple-peak")


## 3. The explicit cost model

Every multi-controlled X gate is expanded with a named construction from the literature before anything is counted,
so ancillas and T gates come from explicit circuits. The drawings show three constructions for four controls:

- the **throwaway tree** of the manuscript's Figure 1, which leaves its ancillas in a garbage state and is valid only
  when the gate is later undone by its mirror;
- the **restoring tree**, which uncomputes its intermediate ANDs with cheap relative-phase Toffoli gates;
- **Khattar and Gidney's** construction with two clean ancillas, which uses conditionally clean ancillas.

In [ ]:
gate = QuantumCircuit(5)
gate.mcx([0, 1, 2, 3], 4)
for method in ("throwaway_tree", "clean_tree", "2_clean_kg24"):
    circuit, info = costs.decompose_mcx(gate, method)
    display(Markdown(f"**{method}** ({info['mcx_reference']}): {info['appended_ancillas']} ancilla(s)"))
    show_figure(circuit.draw("mpl", fold=40), f"mcx-{method}")

The chart compares the constructions as the number of controls grows. Ancilla-free constructions pay heavily in T
gates; constructions with a few clean ancillas stay close to the restoring tree, which uses n − 2 ancillas.

In [ ]:
shown = ["throwaway_tree", "clean_tree", "2_clean_kg24", "1_clean_kg24", "n_dirty_i15", "noaux_v24"]
isolated = pd.DataFrame(
    [costs.isolated_mcx_costs(method, n) for method in shown for n in (3, 4, 6, 8, 12, 16, 20)]
)
figure, axes = plt.subplots(1, 2, figsize=(10, 3.4))
for color, method in zip(SERIES, shown):
    part = isolated[(isolated.method == method) & (isolated.controls >= 4)]
    axes[0].plot(part.controls, part.t_count, marker="o", markersize=4, linewidth=2, color=color, label=method)
    axes[1].plot(part.controls, part.ancillas, marker="o", markersize=4, linewidth=2, color=color, label=method)
axes[0].set_yscale("log")
log_axis(axes[0].yaxis)
axes[0].set_ylabel("T gates")
axes[1].set_ylabel("Ancillas")
for ax in axes:
    ax.set_xlabel("Controls")
axes[1].legend(frameon=False, loc="upper left", fontsize=8)
show_figure(figure, "mcx-isolated")
display(Markdown("T gates by construction and number of controls"))
display(isolated.pivot(index="controls", columns="method", values="t_count")[shown])
display(Markdown("Arbitrary rotations left by the ancilla-free construction (priced separately)"))
display(isolated[isolated.method == "noaux_v24"].set_index("controls")[["rotation_count"]].T)

Counting uses fixed gadgets with exact T-counts: a Toffoli gate costs 7 T gates, and a relative-phase Toffoli gate
(or the doubly controlled Rx(±π) produced by XAG synthesis) costs 4. The cheaper gadget is allowed only where its
extra phase provably cancels; the same rules are applied to every method, so no flow gets a discount the others do
not. Uncomputing an AND gate by measurement (Gidney) costs no T gate at all; the table shows how much this saves on the
oracle, reported separately because it needs mid-circuit measurement. Every compute–uncompute pair is detected the same way for every method, including pairs whose target is held at 1 or flipped and restored in between.


In [ ]:
oracles = rows[(rows.component == "oracle") & rows.metric_level.isin(["fault-tolerant", "fault-tolerant, measurement-assisted"])]
pivot = oracles[(oracles.regime == "R3") & (oracles.mcx_policy == "clean_tree")].pivot_table(
    index=["benchmark", "method"], columns="metric_level", values="t_count"
)
pivot["saving"] = 1 - pivot["fault-tolerant, measurement-assisted"] / pivot["fault-tolerant"]
savings = pivot["saving"].unstack().reindex(index=GRAPHS)[["xag", "aig_bennett", "xag_bennett", "sb_simple", "sb_balanced"]]
display(Markdown("Share of the oracle's T gates saved by measurement-based uncomputation (R3, restoring tree)"))
display(savings.round(2))
KEY["measurement saving"] = savings.max().round(2).to_dict()

## 4. The corrected comparison

Thirteen constructions (nine synthesis methods and four baselines) were built on the nine graphs in the three
settings, every multi-controlled gate was expanded with each of five constructions, and every circuit was priced in
Clifford+T. A search is compared with the **strongest baseline** on the same graph and setting: a ratio below 1 means
fewer T gates or qubits than the best baseline. Each method uses its own best decomposition.

Three metrics are compared per Grover iteration (the oracle followed by the diffusion operator): T gates, T-depth and
qubits. T-depth adds the depths of the two components and counts each synthesized rotation on the critical path at its
T-count, an upper estimate. The strongest baseline is chosen **separately for each metric, graph and setting**, so the
baseline behind a T ratio and the one behind a qubit ratio can differ; the second table names them. The narrowest
baseline is usually Belletti-Minimal with Vale et al.'s ancilla-free decomposition, which needs thousands of
arbitrary rotations; the column *qubits, practical* therefore also compares with the narrowest baseline among the
decompositions without rotations (throwaway and clean trees, Khattar–Gidney, Iten et al.). Belletti-Minimal stays the narrowest even
then, at up to 20 times the T gates, so the column *qubits, comparable T* compares each method with the narrowest
baseline whose T gates per iteration stay within 1.5 times the method's own. These are the
oracles as synthesized for the study, without the shapes and scheduling of Section 6.


In [ ]:
per_iteration = analysis.iteration_costs(rows, budget=settings.get("rotation_budget", 1e-3))
comparisons = {
    metric: analysis.compare_with_baselines(per_iteration, metric, matched_policy=False)
    for metric in ("t_count", "t_depth", "qubits_total")
}
# The narrowest baseline among decompositions that need no arbitrary rotations.
ROTATION_FREE = [p for p in POLICIES if p != "noaux_v24"]
practical = per_iteration[~per_iteration.method.str.startswith("sb_") | per_iteration.mcx_policy.isin(ROTATION_FREE)]
comparisons["qubits_practical"] = analysis.compare_with_baselines(practical, "qubits_total", matched_policy=False)
# The narrowest baseline whose T gates per iteration stay within T_FACTOR times the method's
# (each method in its configuration with the fewest T gates).
T_FACTOR = 1.5
keys = ["benchmark", "regime", "iterations", "method"]
cheapest = per_iteration.loc[per_iteration.groupby(keys).t_count.idxmin()]
pool = per_iteration[per_iteration.method.str.startswith("sb_")]
comparable = []
for _, m in cheapest[~cheapest.method.str.startswith("sb_")].iterrows():
    candidates = pool[(pool.benchmark == m.benchmark) & (pool.regime == m.regime) & (pool.t_count <= T_FACTOR * m.t_count)]
    if candidates.empty:
        continue
    b = candidates.loc[candidates.qubits_total.idxmin()]
    ratio = m.qubits_total / b.qubits_total
    comparable.append({"benchmark": m.benchmark, "regime": m.regime, "iterations": m.iterations, "method": m.method,
                       "mcx_policy": m.mcx_policy, "qubits_total": m.qubits_total, "baseline": b.method,
                       "baseline_policy": b.mcx_policy, "baseline_value": b.qubits_total, "ratio": ratio,
                       "outcome": "win" if ratio < 0.99 else "loss" if ratio > 1.01 else "tie"})
comparisons["qubits_comparable_t"] = pd.DataFrame(comparable)
summary = pd.concat(
    {metric: analysis.summarize_outcomes(table) for metric, table in comparisons.items()}, names=["metric"]
).reset_index()
view = summary.pivot_table(index="method", columns=["metric", "regime"], values="geometric_mean_ratio").reindex(LOGIQUE)
display(Markdown("Geometric mean over the nine graphs of the ratio to the strongest baseline, per Grover iteration"))
display(view.round(2))
outcomes = summary[summary.method == "xag"].set_index(["metric", "regime"])[["wins", "ties", "losses"]]
display(Markdown("Wins, ties and losses of the XAG oracle against the strongest baseline"))
display(outcomes)
KEY["xag ratios"] = view.loc["xag"].round(2).to_dict()

In [ ]:
NAMES = {"sb_original": "Saha et al.", "sb_simple": "Belletti-Simple", "sb_minimal": "Belletti-Minimal",
         "sb_balanced": "Belletti-Balanced"}
POLICY_NAMES = {"throwaway_tree": "throwaway tree", "clean_tree": "clean tree", "2_clean_kg24": "Khattar–Gidney",
                "n_dirty_i15": "Iten et al. (dirty)", "noaux_v24": "Vale et al. (no ancilla)"}
METRICS = {"t_count": "T gates", "t_depth": "T-depth", "qubits_total": "qubits",
           "qubits_practical": "qubits, practical", "qubits_comparable_t": "qubits, comparable T"}
named = []
for metric, label in METRICS.items():
    part = comparisons[metric][comparisons[metric].method == "xag"]
    column = "qubits_total" if metric.startswith("qubits") else metric
    for _, r in part.iterrows():
        named.append({
            "regime": r.regime, "graph": r.benchmark, "metric": label,
            "XAG": f"{r[column]:,.0f}",
            "strongest baseline": f"{NAMES[r.baseline]}, {POLICY_NAMES[r.baseline_policy]}",
            "baseline": f"{r.baseline_value:,.0f}",
            "ratio": round(r.ratio, 2),
        })
named = pd.DataFrame(named)
named.to_csv(OUT / "xag_against_strongest_baseline.csv", index=False)
for regime in REGIMES:
    table = (named[named.regime == regime].drop(columns="regime")
             .pivot(index="graph", columns="metric").reindex(GRAPHS))
    table = table.reorder_levels([1, 0], axis=1)[[(m, c) for m in METRICS.values()
                                                  for c in ("XAG", "strongest baseline", "baseline", "ratio")]]
    display(Markdown(f"**{regime}: the XAG oracle per iteration against the strongest baseline for each metric**"))
    display(table)

In [ ]:
diverging = LinearSegmentedColormap.from_list("ratio", [SERIES[0], "#f0efec", SERIES[7]])
table = comparisons["t_count"]
figure, axes = plt.subplots(1, len(REGIMES), figsize=(4.3 * len(REGIMES), 0.42 * len(LOGIQUE) + 1.6), sharey=True)
for ax, regime in zip(np.atleast_1d(axes), REGIMES):
    grid = table[table.regime == regime].pivot(index="method", columns="benchmark", values="ratio")
    grid = grid.reindex(index=LOGIQUE, columns=[g for g in GRAPHS if g in grid.columns])
    ax.imshow(np.log2(grid.to_numpy(dtype=float)), cmap=diverging, norm=TwoSlopeNorm(0, -4, 4), aspect="auto")
    for (i, j), ratio in np.ndenumerate(grid.to_numpy(dtype=float)):
        if np.isfinite(ratio):
            ax.text(j, i, f"{ratio:.2f}" if ratio < 10 else f"{ratio:.0f}", ha="center", va="center", fontsize=7, color=INK)
    ax.set_xticks(range(grid.shape[1]), grid.columns, rotation=45, ha="right")
    ax.set_yticks(range(grid.shape[0]), grid.index)
    ax.set_title(regime, color=INK)
    ax.grid(False)
figure.suptitle("T gates per iteration relative to the strongest baseline (blue: fewer, red: more)", color=INK, y=1.02)
show_figure(figure, "t-ratio-heatmap")

The next chart isolates the XAG oracle, Logique's main flow, per iteration: its T-count never exceeds the strongest
baseline, its T-depth is higher on most graphs (before the shapes and scheduling of Section 6), and it needs more
qubits on every graph, and still 1.4 to 3.6 times as many as the narrowest baseline at comparable T.


In [ ]:
figure, axes = plt.subplots(1, 5, figsize=(20, 3.4), sharey=False)
for ax, metric, title in zip(axes, ("t_count", "t_depth", "qubits_total", "qubits_practical", "qubits_comparable_t"), ("T gates", "T-depth", "Qubits", "Qubits, practical", "Qubits, comparable T")):
    part = comparisons[metric][comparisons[metric].method == "xag"]
    for r, regime in enumerate(REGIMES):
        values = part[part.regime == regime].set_index("benchmark").ratio.reindex(GRAPHS)
        ax.scatter(np.arange(len(GRAPHS)) + (r - 1) * 0.18, values, s=40, color=SERIES[r], edgecolor="white", linewidth=0.8, label=regime, zorder=2)
    ax.axhline(1, color=SECONDARY, linewidth=1, zorder=1)
    ax.set_xticks(range(len(GRAPHS)), GRAPHS, rotation=45, ha="right")
    ax.set_title(f"{title}: XAG / strongest baseline", color=INK)
    ax.grid(axis="x", visible=False)
axes[0].legend(frameon=False, loc="lower left")
show_figure(figure, "xag-vs-baseline")

Where do the qubits go? The bars split each circuit's width into the data qubits, the output qubit, the oracle's own
workspace and the ancillas added by the decomposition (edge-only setting, Khattar–Gidney decomposition).

In [ ]:
widths_by_role = analysis.width_breakdown(rows)
policy = "2_clean_kg24" if "2_clean_kg24" in POLICIES else POLICIES[0]
shown_graphs = ["K5", "K10", "myciel4", "K20"]
methods = ["xag", "xag_lowt", "aig_bennett", *BASELINES]
roles = {"data_qubits": "data", "output_qubits": "output", "oracle_workspace": "oracle workspace", "mcx_ancillas": "decomposition ancillas"}
part = widths_by_role[(widths_by_role.regime == "R3") & (widths_by_role.mcx_policy == policy)]
figure, axes = plt.subplots(1, len(shown_graphs), figsize=(12, 3.2), sharey=True)
for ax, g in zip(axes, shown_graphs):
    data = part[part.benchmark == g].set_index("method").reindex(methods).fillna(0)
    left = np.zeros(len(data))
    for color, (role, label) in zip(SERIES, roles.items()):
        ax.barh(data.index, data[role], left=left, color=color, edgecolor="white", height=0.7, label=label)
        left += data[role].to_numpy()
    ax.invert_yaxis()
    ax.set_title(g, color=INK)
    ax.grid(axis="y", visible=False)
handles, names = axes[0].get_legend_handles_labels()
figure.legend(handles, names, frameon=False, ncol=4, loc="lower center", bbox_to_anchor=(0.5, -0.08))
figure.tight_layout()
show_figure(figure, "width-by-role")

## 5. Why our oracle ties Belletti-Simple

When only edges are checked, Logique's XAG oracle and the Belletti-Simple baseline are the same circuit. Both compare
each edge's colors with c − 1 AND gates and combine the |E| results with |E| − 1 more. Once Belletti-Simple's
multi-controlled gates are expanded with the manuscript's throwaway tree, the two have the same Toffoli gates, the
same T-count, 8(|E|c − 2) + 7, and the same width, (|V| + |E|)c − 1 qubits. The drawings show K₃: the registers `v`
hold the colors, `w` and `e` hold intermediate results, `a` the tree's ancilla and `out` the output.

In [ ]:
xag_k3 = logique_instance(replace(problems["K3"]), "xag", "R3", executable=EXECUTABLE)
simple_k3 = baseline_instance(problems["K3"], "simple", "R3")
for name, instance in (("XAG oracle", xag_k3), ("Belletti-Simple, throwaway tree", simple_k3)):
    circuit = expanded(instance)
    display(Markdown(f"**{name}**: {circuit.num_qubits} qubits, {costs.ft_counts(circuit)['t_count']} T gates"))
    show_figure(labelled(circuit, role_groups(instance, circuit)).draw("mpl", fold=45), f"k3-{instance.method}")

In [ ]:
equivalence = []
for g in GRAPHS:
    p = problems[g]
    c = max(1, (p.colors - 1).bit_length())
    pick = lambda m: ft[(ft.component == "oracle") & (ft.benchmark == g) & (ft.regime == "R3") & (ft.method == m) & (ft.mcx_policy == "throwaway_tree")].iloc[0]
    x, s = pick("xag"), pick("sb_simple")
    balanced = expanded(logique_instance(p, "xag", "R3", executable=EXECUTABLE, balanced=True))
    equivalence.append(
        {
            "graph": g,
            "T, XAG": int(x.t_count),
            "T, Belletti-Simple": int(s.t_count),
            "T, formula": 8 * (len(p.edges) * c - 2) + 7,
            "qubits, XAG": int(x.qubits),
            "qubits, Belletti-Simple": int(s.qubits),
            "qubits, formula": (len(p.vertices) + len(p.edges)) * c - 1,
            "T-depth, XAG": int(x.t_depth),
            "T-depth, XAG balanced": int(costs.ft_metrics(costs.clifford_t(balanced))["t_depth"]),
            "T-depth, Belletti-Simple": int(s.t_depth),
        }
    )
equivalence = pd.DataFrame(equivalence).set_index("graph")
display(equivalence)
assert (equivalence["T, XAG"] == equivalence["T, Belletti-Simple"]).all()
assert (equivalence["qubits, XAG"] == equivalence["qubits, Belletti-Simple"]).all()

The counts are identical; only the order of the gates differs. The XAG joins the edge results in a long chain, which
makes it deeper; building the conjunction as a balanced tree removes most of that difference. The width claims of the
manuscript therefore came from the ancilla counting of Section 2, not from the synthesis. Worse for Logique,
Belletti-Simple keeps its multi-controlled gates until decomposition, so other constructions can reuse ancillas and
make it narrower, at a moderate cost in T gates; the synthesized circuit fixes one qubit per AND gate.

In [ ]:
oracle_r3 = ft[(ft.component == "oracle") & (ft.regime == "R3")]
xag_r3 = oracle_r3[oracle_r3.method == "xag"].groupby("benchmark")[["qubits", "t_count"]].min()
simple_r3 = oracle_r3[oracle_r3.method == "sb_simple"].set_index(["benchmark", "mcx_policy"])[["qubits", "t_count"]]
trade = simple_r3.div(xag_r3, level="benchmark").reset_index()
figure, axes = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
order = list(reversed(GRAPHS))
for ax, column, title in zip(axes, ("qubits", "t_count"), ("qubits", "T gates")):
    for offset, (policy_name, color) in zip(np.linspace(0.24, -0.24, len(POLICIES)), zip(POLICIES, SERIES)):
        values = trade[trade.mcx_policy == policy_name].set_index("benchmark")[column].reindex(order)
        ax.scatter(values, np.arange(len(order)) + offset, s=30, color=color, label=policy_name, zorder=2)
    ax.axvline(1, color=SECONDARY, linewidth=1)
    ax.set_xscale("log")
    log_axis(ax.xaxis)
    ax.set_xlabel(f"Belletti-Simple {title} / XAG")
    ax.grid(axis="y", visible=False)
axes[0].set_yticks(range(len(order)), order)
handles, names = axes[0].get_legend_handles_labels()
figure.legend(handles, names, frameon=False, ncol=5, loc="lower center", bbox_to_anchor=(0.5, -0.1))
figure.tight_layout()
show_figure(figure, "simple-policies")
KEY["simple narrowest"] = float(trade.qubits.min())

## 6. Optimizing the synthesized oracle

Three optimizations act on the synthesized oracle before it is mapped to gates. **Multiplicative-complexity
reduction** rewrites the logic with fewer AND gates; it helps where the predicate is more than a set of edge
comparisons, namely the checks of unused codes. **Balanced conjunctions** keep the gates but shorten the circuit.
**Caterpillar's mapping strategies** decide which intermediate results are kept or recomputed.

In [ ]:
reduction = []
for g in GRAPHS:
    if g in ("K20", "myciel5"):
        continue  # the pass is slow on the largest predicates
    encoding = encode_coloring(problems[g])
    plain = synthesize(encoding, "xag", executable=EXECUTABLE).summary()["AND"]
    reduced = synthesize(encoding, "xag", executable=EXECUTABLE, mc_reduction=True).summary()["AND"]
    reduction.append({"graph": g, "AND gates": plain, "after the pass": reduced, "removed": plain - reduced})
display(Markdown("AND gates of the complete predicate (R1 and R2) before and after the reduction pass"))
display(pd.DataFrame(reduction).set_index("graph"))

strategies = comparisons["t_count"]
strategy_view = pd.concat(
    {
        metric: comparisons[metric][comparisons[metric].regime == "R3"].pivot(index="benchmark", columns="method", values="ratio")
        for metric in ("t_count", "qubits_total")
    },
    axis=1,
).reindex(GRAPHS)
methods = [m for m in ("xag", "xag_lowt", "xag_fast_lowt", "xag_lowdepth", "xag_eager", "xag_bennett", "aig_bennett") if m in LOGIQUE]
display(Markdown("Caterpillar's strategies against the strongest baseline per iteration (R3): T gates, then qubits"))
display(strategy_view.loc[:, (slice(None), methods)].round(2))

### Shape of the final conjunction and gate scheduling

When only edges are checked, the XAG oracle and Belletti-Simple use the same AND gates, but the XAG was 1.1 to 6 times
deeper in T. Two causes add up, and neither changes a gate or a qubit.

- **Gate order.** Qiskit counts any two gates on a shared wire as ordered, even when they commute, such as Toffoli
  gates that share only a control. The synthesized circuit lists the edge comparisons in input order, so comparisons
  that share a vertex run one after the other. A commutation-aware list scheduler reorders the compute block (gates
  swap when they act in the same basis on every shared wire), keeps the uncompute block its exact inverse, and keeps
  the new order only if it prices to a lower T-depth.
- **Shape of the final conjunction.** A balanced tree is only optimal when all its inputs are ready together. The
  encoder now offers three shapes with the same gates: *input* pairs the constraints in input order; *rounds* orders
  the edges by an edge colouring, so that paired edges share no vertex; *timed* always joins the two constraints
  expected to be ready first, as in a Huffman tree over ready times.

The drawings show the three shapes on a small graph (the bull graph with three colors), each scheduled: the AND network
of the edge-only predicate, then the oracle.

In [ ]:
from logique.visualization.boolean import graphviz_graph
from logique.benchmarks.scheduling import priced_t_depth
from logique.coloring.encoding import TREES
from logique.coloring.model import ColoringProblem

bull = ColoringProblem((0, 1, 2, 3, 4), ((0, 1), (1, 2), (0, 2), (1, 3), (2, 4)), 3)
for tree in TREES:
    shaped = logique_instance(bull, "xag", "R3", executable=EXECUTABLE, balanced=True, tree=tree)
    tuned = logique_instance(bull, "xag", "R3", executable=EXECUTABLE, balanced=True, tree=tree, schedule=True)
    network = synthesize(tuned.encoding, "xag", executable=EXECUTABLE).metadata["network"]
    labels = [None] * tuned.encoding.input_count
    for vertex, inputs in tuned.encoding.input_mapping.items():
        for bit, i in enumerate(inputs):
            labels[i] = f"x_{vertex},{bit}"
    lowered = costs.ft_metrics(costs.clifford_t(costs.paired_relative_phase_toffolis(tuned.oracle)))
    display(Markdown(
        f"**Tree shape `{tree}`**: {tuned.oracle.num_qubits} qubits, {int(lowered['t_count'])} T gates, T-depth "
        f"{priced_t_depth(shaped.oracle)} as synthesized and {int(lowered['t_depth'])} scheduled"
        f"{'' if tuned.structure['scheduled'] else ' (the synthesized order was kept)'}."
    ))
    show_svg(graphviz_graph(network, f"Edge-only predicate of the bull graph, tree '{tree}'", input_labels=labels).pipe(format="svg"),
             f"bull-network-{tree}")
    show_figure(labelled(tuned.oracle, role_groups(tuned, tuned.oracle)).draw("mpl", fold=60), f"bull-oracle-{tree}")

The tables repeat the experiment on the nine graphs in all three settings, from the study runs of the three shapes
(oracle only, unitary column). Every shape keeps the qubits and T gates of the balanced XAG oracle; only the T-depth
changes. Belletti-Simple is rescheduled the same way, so the comparison is fair. With the Hadamard or valid-color start
the oracle also checks the unused codes, which the XAG flow synthesizes with fewer T gates than the baselines; the T and
qubit columns show that side of the comparison.


In [ ]:
SHAPES = {tree: analysis.load_study(latest_run(f"tree-shapes-{tree}"))[0] for tree in TREES}
SCHEDULED_BASELINES = analysis.load_study(latest_run("scheduled-baselines"))[0]


def oracle_cost(data, method, regime, policy="throwaway_tree"):
    """Unitary oracle costs of one method, regime and policy, indexed by graph."""
    d = data[(data.component == "oracle") & (data.metric_level == "fault-tolerant") & (data.method == method)
             & (data.regime == regime) & (data.mcx_policy == policy)]
    return d.set_index("benchmark").reindex(GRAPHS)


shape_tables = {}
figure, axes = plt.subplots(1, len(REGIMES), figsize=(6 * len(REGIMES), 3.6), sharey=True)
for ax, regime in zip(axes, REGIMES):
    study_xag = oracle_cost(rows, "xag", regime)
    shaped = {tree: oracle_cost(SHAPES[tree], "xag", regime) for tree in TREES}
    simple = oracle_cost(SCHEDULED_BASELINES, "sb_simple", regime)
    simple_kg = oracle_cost(SCHEDULED_BASELINES, "sb_simple", regime, "2_clean_kg24")
    table = pd.DataFrame(index=GRAPHS)
    table["XAG as in the study"] = study_xag.t_depth
    for tree in TREES:
        table[f"XAG, {tree}"] = shaped[tree].t_depth
    table["XAG, best shape"] = table[[f"XAG, {tree}" for tree in TREES]].min(axis=1)
    table["B-S, throwaway"] = simple.t_depth
    table["B-S, Khattar–Gidney"] = simple_kg.t_depth
    table["T: XAG"] = shaped["input"].t_count
    table["T: B-S"] = simple.t_count
    table["qubits: XAG"] = shaped["input"].qubits_total
    table["qubits: B-S"] = simple.qubits_total
    table = table.astype(int)
    shape_tables[regime] = table
    same = all((shaped[t].qubits_total == shaped["input"].qubits_total).all()
               and (shaped[t].t_count == shaped["input"].t_count).all() for t in TREES)
    ahead = (table["XAG, best shape"] < table["B-S, throwaway"]).sum()
    tied = (table["XAG, best shape"] == table["B-S, throwaway"]).sum()
    display(Markdown(f"**{regime}: oracle T-depth by conjunction shape, scheduled** (T and qubits at the right; "
                     f"the three shapes {'share' if same else 'do not all share'} the same qubits and T gates)"))
    display(table)
    display(Markdown(f"With the best shape, the XAG oracle is shallower than Belletti-Simple (throwaway tree, scheduled) on "
                     f"{ahead} of {len(GRAPHS)} graphs, tied on {tied} and deeper on {len(GRAPHS) - ahead - tied}; it uses "
                     f"{(table['T: XAG'] / table['T: B-S']).min():.2f}× to {(table['T: XAG'] / table['T: B-S']).max():.2f}× "
                     f"Belletti-Simple's T gates."))
    columns = ["XAG as in the study", "XAG, best shape", "B-S, throwaway"]
    x = np.arange(len(GRAPHS))
    for k, (column, color) in enumerate(zip(columns, [SERIES[1], SERIES[0], SERIES[3]])):
        ax.bar(x + (k - 1) * 0.27, table[column], width=0.27, color=color, edgecolor="white", label=column)
    ax.set_yscale("log")
    log_axis(ax.yaxis)
    ax.set_xticks(x, GRAPHS, rotation=30, ha="right")
    ax.set_title(regime, color=INK)
    ax.grid(axis="x", visible=False)
axes[0].set_ylabel("T-depth of the oracle")
axes[0].legend(frameon=False, loc="upper left")
show_figure(figure, "tree-shapes")
depth_table = shape_tables["R3"]

### T-depth of every method, before and after optimization

The table gives the oracle's T-depth for all thirteen constructions, each with its shallowest decomposition, in three
versions: as in the nine-graph study; with the balanced encoder; and with the balanced encoder and gate scheduling
(for the XAG flow, with the best of the three conjunction shapes). The baselines do not use the encoder, so their
balanced version is the study's; their scheduled version reorders the same gates. Each cell reads *study → balanced →
scheduled*. The versions come from separate study runs, located by name; a run still in progress shows as pending,
and the cell can be rerun when it finishes. `LEVEL` switches between the unitary and the measurement-based column.


In [ ]:
LEVEL = "fault-tolerant"  # or MEASURED, the measurement-based column


def oracle_depths(source, level=LEVEL):
    """Shallowest oracle T-depth over the decomposition policies, per graph, method and regime."""
    if source is None:
        return pd.Series(dtype=float)
    data = source if isinstance(source, pd.DataFrame) else analysis.load_study(source)[0]
    data = data[(data.component == "oracle") & (data.metric_level == level)]
    return data.groupby(["benchmark", "method", "regime"]).t_depth.min()


runs = {
    "study": rows,  # the study, with the repriced measurement-based rows when available
    "balanced": latest_run("balanced-methods"),
    "scheduled": latest_run("scheduled-methods"),
    "scheduled baselines": latest_run("scheduled-baselines"),
    **{f"shape {tree}": latest_run(f"tree-shapes-{tree}") for tree in TREES},
}
display(pd.DataFrame({"run": {
    k: (RUN.name if isinstance(v, pd.DataFrame) else v.name if v else "pending") for k, v in runs.items()
}}))
depths = {name: oracle_depths(run) for name, run in runs.items()}
study_depth = depths["study"]
methods = LOGIQUE + BASELINES
index = pd.MultiIndex.from_product([GRAPHS, methods, REGIMES], names=["benchmark", "method", "regime"])
versions = pd.DataFrame(index=index)
versions["study"] = study_depth.reindex(index)
balanced = pd.concat([depths["balanced"], study_depth[study_depth.index.get_level_values("method").isin(BASELINES)]])
versions["balanced"] = balanced.reindex(index)
shapes_best = pd.concat([depths[f"shape {tree}"] for tree in TREES], axis=1).min(axis=1) if any(
    len(depths[f"shape {tree}"]) for tree in TREES) else pd.Series(dtype=float)
scheduled = pd.concat([depths["scheduled"], depths["scheduled baselines"], shapes_best])
versions["scheduled"] = scheduled[~scheduled.index.duplicated()].reindex(index)
# The largest best-fit oracles exceed the study's pricing limit (3 million gates) and were priced again with a
# higher limit; their depths fill the gaps.
FILL = {"study": "best-fit-depths", "balanced": "best-fit-depths-balanced", "scheduled": "best-fit-depths-scheduled"}
for version, description in FILL.items():
    versions[version] = versions[version].fillna(oracle_depths(latest_run(description)).reindex(index))
versions.reset_index().assign(level=LEVEL).to_csv(OUT / "t_depth_all_methods.csv", index=False)


def cell(row):
    values = ["–" if pd.isna(v) else f"{int(v):,}" for v in row]
    return " → ".join(values)


column = "unitary" if LEVEL == "fault-tolerant" else "measurement-based"
for REGIME in REGIMES:
    chosen = versions.xs(REGIME, level="regime")
    table = chosen.apply(cell, axis=1).unstack("benchmark").reindex(index=methods, columns=GRAPHS)
    display(Markdown(f"**Oracle T-depth under {REGIME}, {column} column** (study → balanced → scheduled; – means not priced)"))
    display(table)

    ratio = (chosen["scheduled"] / chosen["study"]).unstack("benchmark").reindex(index=methods, columns=GRAPHS)
    figure, ax = plt.subplots(figsize=(9, 5))
    shades = LinearSegmentedColormap.from_list("gain", [SERIES[2], "#f0efec", SERIES[7]])
    image = ax.imshow(ratio.astype(float).values, cmap=shades, norm=TwoSlopeNorm(vcenter=1, vmin=0, vmax=2), aspect="auto")
    ax.set_xticks(range(len(GRAPHS)), GRAPHS, rotation=30, ha="right")
    ax.set_yticks(range(len(methods)), methods)
    for (i, j), value in np.ndenumerate(ratio.astype(float).values):
        if value == value:
            ax.text(j, i, f"{value:.2f}", ha="center", va="center", fontsize=7, color=INK)
    ax.grid(False)
    ax.set_title(f"Oracle T-depth after optimization ÷ before, {REGIME} (blank: pending)")
    figure.colorbar(image, ax=ax, fraction=0.03)
    show_figure(figure, f"t-depth-all-methods-{REGIME}")

### Measurement-based uncomputation, priced fairly

Uncomputing an AND by measurement removes the T gates of the uncompute half. Both the XAG oracle and Belletti-Simple
compute the same AND gates in mirrored blocks, so both should save the same. On K₃ the corrected detector pairs four
ANDs in each (the gate on the output stays a Toffoli); the earlier detector paired four in the XAG oracle but only one
in Belletti-Simple, whose flags are set to |1⟩ and wrapped in X gates around the open controls of its conjunction, which
made the XAG look 34% cheaper. The table below checks K₃ with the corrected pricing; the tables compare the optimized XAG oracle (best
conjunction shape, scheduled) with Belletti-Simple (throwaway tree, scheduled) in both columns, in all three settings.

In [ ]:
from logique.benchmarks import logical_and

k3_rows = []
for name, instance in (
    ("XAG", logique_instance(problems["K3"], "xag", "R3", executable=EXECUTABLE)),
    ("Belletti-Simple, throwaway tree", baseline_instance(problems["K3"], "simple", "R3")),
):
    circuit = expanded(instance)
    clean = [*instance.clean, *range(instance.oracle.num_qubits, circuit.num_qubits)]
    unitary = costs.ft_metrics(costs.clifford_t(circuit))
    measured = logical_and.measurement_assisted_metrics(circuit, clean_wires=clean)
    k3_rows.append({"oracle": name, "qubits": circuit.num_qubits, "ANDs uncomputed by measurement": measured["and_pairs"],
                    "T, unitary": int(unitary["t_count"]), "T, measurement-based": int(measured["t_count"]),
                    "T-depth, unitary": int(unitary["t_depth"]), "T-depth, measurement-based": int(measured["t_depth"]),
                    "feedforward depth": measured["reaction_depth"]})
display(pd.DataFrame(k3_rows).set_index("oracle"))


def oracle_in(source, method, regime, policy=None):
    data = analysis.load_study(source)[0] if not isinstance(source, pd.DataFrame) else source
    data = data[(data.component == "oracle") & (data.regime == regime) & (data.method == method)]
    return data if policy is None else data[data.mcx_policy == policy]


shape_data = [analysis.load_study(latest_run(f"tree-shapes-{tree}"))[0] for tree in TREES]
simple_data = analysis.load_study(latest_run("scheduled-baselines"))[0]
for regime in REGIMES:
    xag_best = pd.concat([oracle_in(d, "xag", regime) for d in shape_data])
    simple = oracle_in(simple_data, "sb_simple", regime, "throwaway_tree")
    fair_rows = []
    for g in GRAPHS:
        row = {"graph": g}
        for level, column in (("fault-tolerant", "unitary"), (MEASURED, "measurement")):
            x = xag_best[(xag_best.benchmark == g) & (xag_best.metric_level == level)]
            s = simple[(simple.benchmark == g) & (simple.metric_level == level)]
            row[f"T {column}: XAG"] = int(x.t_count.min())
            row[f"T {column}: B-S"] = int(s.t_count.min())
            row[f"T-depth {column}: XAG"] = int(x.t_depth.min())
            row[f"T-depth {column}: B-S"] = int(s.t_depth.min())
        fair_rows.append(row)
    fair = pd.DataFrame(fair_rows).set_index("graph")
    display(Markdown(f"**{regime}**: XAG oracle with the best conjunction shape, scheduled, against Belletti-Simple "
                     f"(throwaway tree, scheduled), in both pricing columns"))
    display(fair)


## 7. Cheaper preparation of the valid colors

For every number of colors in the benchmarks (3, 4, 5, 6, 10 and 20), the preparation of a vertex needs **one**
y-rotation, which splits the amplitude 1 : 2 (k = 3 and 6) or 1 : 4 (k = 5, 10 and 20), plus Hadamard and controlled
Hadamard gates. The preparation and its inverse run in every iteration, so this rotation is paid 2j + 1 times per
vertex, and each copy must be approximated with Clifford+T gates to a share of the error budget. Four alternatives were
implemented and verified:

- **Exact rotations by repeat-until-success.** Both rotations can be implemented *exactly* by a small circuit with
  ancillas and a measurement: on success the rotation is applied, on failure the qubit is left unchanged and the
  attempt is repeated. For 1 : 2, an exhaustive search found an attempt with only 2 T gates that succeeds with
  probability 3/4; for 1 : 4, Jones's attempt uses one AND gate (4 T) and succeeds with probability 5/8. Exact
  rotations take no share of the error budget.
- **Better approximations** of each rotation, priced with the mean T-counts reported by Kliuchnikov et al. (2023):
  mixed diagonal, fallback (repeat-until-success), and mixed fallback, against Ross and Selinger's unitary method.
- **Hamming-weight phasing** (Gidney): the same rotation on n vertices costs about log₂ n rotations plus an adder
  network, using qubits of the oracle's workspace, which are free while the preparation runs.
- **Amplitude-amplified preparation** (Babbush et al.): Hadamard gates plus one exact round of amplitude
  amplification. It needs three rotations per vertex and a comparator, so it is not expected to be cheaper; it is
  included to check.

In [ ]:
rotation_rows = []
for k in range(2, 21):
    circuit = prep.shukla_vedula(k)
    exact = prep.with_exact_rotations(circuit)
    rus = [op for op in exact.count_ops() if op.startswith("rus")]
    rotation_rows.append(
        {
            "colors k": k,
            "rotations": costs.ft_metrics(costs.clifford_t(circuit))["rotation_count"],
            "exact circuit": ", ".join(rus) or ("none needed" if k & (k - 1) == 0 else "no"),
            "expected T (exact)": round(sum(prep.EXACT_ROTATIONS[int(r[6])].expected_t for r in rus), 2) if rus else None,
            "T per approximated rotation (Ross-Selinger, eps=1e-6)": costs.rotation_t_count(1e-6) if k & (k - 1) else None,
        }
    )
display(pd.DataFrame(rotation_rows).set_index("colors k"))

branch_rows = []
for m, rotation in prep.EXACT_ROTATIONS.items():
    unitary = Operator(rotation.attempt).data
    ancillas = rotation.attempt.num_qubits - 1
    for outcome in range(1 << ancillas):
        block = unitary[np.ix_([(outcome << 1) | t for t in (0, 1)], [0, 1])]
        p = float(np.real(np.trace(block.conj().T @ block)) / 2)
        if p > 1e-12:
            label = format(outcome, f"0{ancillas}b")
            branch_rows.append({"split": f"1 : {m - 1}", "ancilla outcome": label, "probability": round(p, 4),
                                "effect": "rotation applied" if label == rotation.success else "target unchanged, retry"})
display(pd.DataFrame(branch_rows))
for m, rotation in prep.EXACT_ROTATIONS.items():
    names = ["target", *[f"ancilla {i}" for i in range(1, rotation.attempt.num_qubits)]]
    groups = [(n.replace(" ", ""), [i]) for i, n in enumerate(names)]
    display(Markdown(f"**One attempt of the 1 : {m - 1} rotation**: {rotation.t_per_attempt} T gates, success "
                     f"probability {rotation.probability}, expected {rotation.expected_t:.2f} T gates. {rotation.reference}."))
    show_figure(labelled(rotation.attempt, groups).draw("mpl"), f"rus-{m}")

In [ ]:
layer = prep.hamming_weight_phasing(5, 0.4)
display(Markdown(f"**Hamming-weight phasing of five vertices**: {layer.count_ops().get('rz', 0)} rotations instead of 5, "
                 f"{prep.hamming_weight_ancillas(5)} ancillas"))
show_figure(layer.draw("mpl", fold=40), "hamming-weight")
counts = pd.DataFrame({"vertices": range(1, 48)})
counts["rotations"] = counts.vertices.map(lambda n: n.bit_length())
counts["ancillas"] = counts.vertices.map(prep.hamming_weight_ancillas)
figure, ax = plt.subplots(figsize=(6, 2.8))
ax.plot(counts.vertices, counts.vertices, color=SERIES[1], linewidth=2, label="direct rotations")
ax.plot(counts.vertices, counts.rotations, color=SERIES[0], linewidth=2, label="Hamming-weight phasing")
ax.set_xlabel("Vertices")
ax.set_ylabel("Rotations per layer")
ax.legend(frameon=False)
show_figure(figure, "hamming-weight-count")

amplified = prep.amplitude_amplified(3)
display(Markdown("**Amplitude-amplified preparation of three colors** (data wires, the extra rotated wire, the comparator flag)"))
show_figure(amplified.draw("mpl", fold=40), "amplitude-amplified")

The decisive test is the complete search. For each graph, the preparation and the diffusion operator are rebuilt with
each method and priced; the oracle is unchanged, so its cost comes from the study. The current preparation with Ross
and Selinger's rotations reproduces the study's totals exactly. Rotations share the error budget of 10⁻³ for the whole
search.

Constructions that need mid-circuit measurement (the exact rotations, and the fallback and mixed-fallback
approximations) are reported in a **separate measurement-based column**, like the measurement-based uncomputation of
Section 3; the main comparison uses unitary pricing only. Mixed diagonal approximation replaces each rotation by a
random mixture of unitaries and forms a third column.

In [ ]:
MODELS = list(costs.ROTATION_SYNTHESIS)
search_rows = []
for g in GRAPHS:
    p = problems[g]
    for regime in ("R2", "R3"):
        instance = logique_instance(p, "xag", regime, executable=EXECUTABLE)
        sel = ft[(ft.benchmark == g) & (ft.regime == regime) & (ft.method == "xag") & (ft.mcx_policy == "clean_tree")]
        oracle_t = float(sel[sel.component == "oracle"].t_count.iloc[0])
        found = sel[sel.component == "search"]
        j = int(found.iterations.iloc[0]) if len(found) else None
        for method in prep.PREPARATIONS:
            preparation_c, diffusion_c, _, clean = prepared.prepared_components(p, instance, method)
            counts_a = prepared.component_counts(preparation_c, "clean_tree", clean)
            counts_d = prepared.component_counts(diffusion_c, "clean_tree", clean)
            for model in MODELS:
                if method == "exact_rus" and model != "ross_selinger":
                    continue  # no approximated rotation is left
                row = {"graph": g, "regime": regime, "pricing": prepared.pricing(method, model), "preparation": method,
                       "rotations": model if method != "exact_rus" else "exact", "qubits": diffusion_c.num_qubits}
                one = prepared.search_total(counts_a, counts_d, oracle_t, j or 1, model=model)
                row["T per iteration"] = (
                    oracle_t
                    + counts_d["t_count"]
                    + counts_d["rus_expected_t"]
                    + counts_d["rotation_count"] * one["t_per_rotation"]
                )
                if j is not None:
                    total = prepared.search_total(counts_a, counts_d, oracle_t, j, model=model)
                    row["T, complete search"] = total["t_total"]
                    row["iterations"] = j
                    if method == "shukla_vedula" and model == "ross_selinger":
                        assert math.isclose(total["t_total"], float(found.t_count.iloc[0])), (g, regime)
                search_rows.append(row)
choices = pd.DataFrame(search_rows)
r1 = search[(search.method == "xag") & (search.regime == "R1") & (search.mcx_policy == "clean_tree")].set_index("benchmark").t_count
current = choices[(choices.preparation == "shukla_vedula") & (choices.rotations == "ross_selinger")].set_index(["graph", "regime"])["T, complete search"]
choices["vs current"] = choices["T, complete search"] / choices.set_index(["graph", "regime"]).index.map(current)
display(Markdown("Complete edge-only search (R3) relative to the current preparation, by pricing column"))
display(
    choices[choices.regime == "R3"]
    .pivot_table(index=["pricing", "preparation", "rotations"], columns="graph", values="vs current")
    .reindex(columns=[g for g in GRAPHS if g in set(choices.dropna(subset=["vs current"]).graph)])
    .round(2)
)

In [ ]:
best_choice = choices.dropna(subset=["T, complete search"])
panels = {
    "Unitary pricing (main comparison)": [
        ("shukla_vedula", "ross_selinger", "current preparation", SERIES[1]),
        ("hamming_weight", "ross_selinger", "Hamming-weight phasing", SERIES[2]),
        ("amplitude_amplified", "ross_selinger", "amplitude-amplified", SERIES[4]),
    ],
    "Measurement-based (separate column)": [
        ("shukla_vedula", "fallback", "fallback approximation", SERIES[6]),
        ("shukla_vedula", "mixed_fallback", "mixed fallback approximation", SERIES[3]),
        ("exact_rus", "exact", "exact rotations", SERIES[0]),
    ],
}
counted = [g for g in GRAPHS if g in set(best_choice.graph)]
rotated = [g for g in counted if problems[g].colors & (problems[g].colors - 1)]
part = best_choice[best_choice.regime == "R2"]
figure, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, (title, series) in zip(axes, panels.items()):
    for i, (method, model, label, color) in enumerate(series):
        values = part[(part.preparation == method) & (part.rotations == model)].set_index("graph")["T, complete search"].reindex(rotated)
        ax.scatter(np.arange(len(rotated)) + (i - 1) * 0.18, values / r1.reindex(rotated).to_numpy(), s=36, color=color,
                   edgecolor="white", linewidth=0.8, label=label, zorder=2)
    ax.axhline(1, color=SECONDARY, linewidth=1, zorder=1, label="Hadamard start (R1)")
    ax.set_yscale("log")
    log_axis(ax.yaxis)
    ax.set_xticks(range(len(rotated)), rotated, rotation=45, ha="right")
    ax.set_title(title, color=INK)
    ax.grid(axis="x", visible=False)
    ax.legend(frameon=False, fontsize=7, loc="lower left")
axes[0].set_ylabel("Complete search (R2) / Hadamard start")
figure.tight_layout()
show_figure(figure, "preparations")

indexed = best_choice.set_index(["preparation", "rotations", "graph", "regime"])["T, complete search"]
saving = lambda method, model: {g: round(1 - indexed[(method, model, g, "R3")] / indexed[("shukla_vedula", "ross_selinger", g, "R3")], 2) for g in rotated}
KEY["exact saving"] = saving("exact_rus", "exact")
KEY["unitary saving"] = saving("hamming_weight", "ross_selinger")
KEY["valid start cheaper"] = {
    column: [g for g in rotated if indexed[(method, model, g, "R2")] < r1[g]]
    for column, (method, model) in (("unitary", ("hamming_weight", "ross_selinger")), ("measurement-based", ("exact_rus", "exact")))
}
winners = best_choice.loc[best_choice.groupby(["graph", "regime", "pricing"])["T, complete search"].idxmin()]
display(Markdown("Cheapest preparation per graph, setting and pricing column"))
display(
    winners[winners.graph.isin(rotated)]
    .set_index(["graph", "regime", "pricing"])[["preparation", "rotations", "T, complete search", "vs current"]]
    .round(3)
)

Graphs whose number of colors is a power of two (myciel3, 2-Insertions_3 and zed-city_2019 have four colors) need no
rotation, so every method costs the same there and they are left out of the chart. In the main, unitary column,
Hamming-weight phasing is the cheapest preparation on every graph that needs rotations; amplitude amplification is more
expensive than the current preparation, as expected from its three rotations per vertex. In the measurement-based column, the exact rotations are cheapest everywhere: they remove the approximated
rotations altogether, at an expected cost of a few T gates per vertex. Starting in valid colors was already cheaper
than the Hadamard start except on K₃; with unitary pricing K₃ still loses, and with the exact rotations it wins too.
What remains of the diffusion operator's cost is its multi-controlled gate over all data qubits.

## 8. Key results and what remains

In [ ]:
ratios = KEY["xag ratios"]
lines = [
    f"- **Corrected comparison.** Per iteration, the XAG oracle needs {ratios[('t_count', 'R1')]:.2f}×, "
    f"{ratios[('t_count', 'R2')]:.2f}× and {ratios[('t_count', 'R3')]:.2f}× the T gates of the strongest baseline "
    f"in R1, R2 and R3 (geometric means), and between "
    f"{min(ratios[('qubits_total', r)] for r in REGIMES):.1f}× and "
    f"{max(ratios[('qubits_total', r)] for r in REGIMES):.1f}× its qubits.",
    "- **Same circuit as Belletti-Simple** when only edges are checked: equal T-count and width on all nine graphs. "
    f"Restoring decompositions make Belletti-Simple as narrow as {KEY['simple narrowest']:.2f}× the XAG oracle.",
    f"- **Measurement-based uncomputation** saves up to {max(KEY['measurement saving'].values()):.0%} of an oracle's T gates.",
    "- **Cheaper preparation, unitary column.** Hamming-weight phasing cuts the complete edge-only search by "
    + ", ".join(f"{g} {v:.0%}" for g, v in KEY["unitary saving"].items())
    + f"; starting in valid colors then beats the Hadamard start on {', '.join(KEY['valid start cheaper']['unitary'])}.",
    "- **Cheaper preparation, measurement-based column.** Exact rotations cut the same search by "
    + ", ".join(f"{g} {v:.0%}" for g, v in KEY["exact saving"].items())
    + f"; starting in valid colors then beats the Hadamard start on {', '.join(KEY['valid start cheaper']['measurement-based'])}"
    + " (graphs with four colors need no rotation).",
    f"- **Diffusion fix.** On K₃, one iteration succeeds with probability "
    f"{KEY['k3 success']['reflection about the preparation']:.2f} instead of {KEY['k3 success']['historical Hadamard reflection']:.2f}.",
]
display(Markdown("\n".join(lines)))

**What remains.** Mapping the synthesized oracle under a qubit budget (SAT-based pebbling), to test whether synthesis
can reach the baseline's width automatically; a balancing tool and a study of all optimization profiles on the nine
graphs; success probability against the number of iterations, with noise; the scalability of synthesis on larger and
denser graphs, with the cost of the complete search set beside the growth of classical algorithms; and the writing of
the revised paper.

## 9. Which configuration wins which metric: the Pareto view

Sections 4 to 6 compare each method with the strongest baseline for one metric at a time, and the strongest baseline
differs by metric: the fewest T gates come from one circuit, the lowest T-depth from another and the fewest qubits from a
third. No single construction wins all three, so the meaningful comparison is the **Pareto frontier**: the
configurations that no other configuration beats on every metric at once. This section gathers every configuration
priced so far, per Grover iteration: the thirteen constructions of the study with each decomposition, and their
optimized versions (balanced encoder, conjunction shapes and gate scheduling). It then shows, per graph, which
configuration wins each metric and which configurations are Pareto-optimal for qubits, T gates and T-depth together.
Configurations with identical costs (an optimization that changes nothing gives the same circuit) count once, and
the tables keep configurations within `T_LIMIT` = 10 times the fewest T gates on each graph, since a circuit with
millions of T gates (Vale et al.'s ancilla-free decomposition, best-fit) is Pareto-optimal only by being the narrowest;
the charts show every configuration. Each setting is shown in turn.

In [ ]:
FAMILIES = {"XAG flow": SERIES[0], "other Logique methods": SERIES[2], "Belletti-Simple": SERIES[1],
            "other baselines": SERIES[4]}
VARIANT_RUNS = {
    "study": rows,
    "balanced": latest_run("balanced-methods"),
    "balanced, scheduled": latest_run("scheduled-methods"),
    "scheduled": latest_run("scheduled-baselines"),
    **{f"shape {tree}, scheduled": latest_run(f"tree-shapes-{tree}") for tree in TREES},
}
REPRICED_DEPTHS = {"study": "best-fit-depths", "balanced": "best-fit-depths-balanced",
                   "balanced, scheduled": "best-fit-depths-scheduled"}
labelled_rows = []
for variant, source in VARIANT_RUNS.items():
    if source is None:
        continue
    data = source if isinstance(source, pd.DataFrame) else analysis.load_study(source)[0]
    data = data[data.metric_level == "fault-tolerant"].copy()
    refill = latest_run(REPRICED_DEPTHS.get(variant, "")) if variant in REPRICED_DEPTHS else None
    if refill is not None:  # best-fit oracles beyond the study's pricing limit, priced again
        extra = analysis.load_study(refill)[0]
        extra = extra[extra.metric_level == "fault-tolerant"]
        replaced = data.set_index(["method", "benchmark", "regime"]).index.isin(
            extra.set_index(["method", "benchmark", "regime"]).index)
        data = pd.concat([data[~replaced], extra], ignore_index=True)
    data["family"] = np.select(
        [data.method == "xag", ~data.method.str.startswith("sb_"), data.method == "sb_simple"],
        ["XAG flow", "other Logique methods", "Belletti-Simple"], "other baselines")
    data["configuration"] = (data.method.map(lambda m: NAMES.get(m, m)) + " [" + variant + ", "
                             + data.mcx_policy.map(POLICY_NAMES) + "]")
    labelled_rows.append(data)
everything = pd.concat(labelled_rows, ignore_index=True)
family = everything.drop_duplicates("configuration").set_index("configuration").family
configs = analysis.iteration_costs(everything.assign(method=everything.configuration).drop(columns="configuration"))
configs = configs.rename(columns={"method": "configuration"}).dropna(subset=["t_depth"])
configs["family"] = configs.configuration.map(family)
PARETO_METRICS = ["qubits_total", "t_count", "t_depth"]
T_LIMIT = 10  # keep configurations within this factor of the fewest T gates on the graph


def pareto(frame):
    """Configurations that no other configuration matches or beats on every metric while beating it on one."""
    values = frame[PARETO_METRICS].to_numpy(float)
    dominated = [((values <= v).all(axis=1) & (values < v).any(axis=1)).any() for v in values]
    return frame[~np.array(dominated)]


def distinct(frame):
    """One row per distinct cost; the label names the first configuration and how many share its cost."""
    grouped = frame.groupby(PARETO_METRICS, sort=False)
    first = grouped.head(1).set_index(PARETO_METRICS)
    first["families"] = grouped.family.agg(lambda f: ", ".join(dict.fromkeys(f)))
    first["identical"] = grouped.size()
    first["configuration"] = [c if n == 1 else f"{c} (+{n - 1} identical)" for c, n in zip(first.configuration, first.identical)]
    return first.reset_index()


for PARETO_REGIME in REGIMES:
    part = configs[configs.regime == PARETO_REGIME]
    winners, frontier = [], []
    for g in GRAPHS:
        graph = part[part.benchmark == g]
        if graph.empty:
            continue
        practical = graph[graph.t_count <= T_LIMIT * graph.t_count.min()]
        row = {"graph": g}
        for metric, label in (("qubits_total", "fewest qubits"), ("t_count", "fewest T gates"), ("t_depth", "lowest T-depth")):
            best = practical[practical[metric] == practical[metric].min()]
            row[label] = f"{best[metric].iloc[0]:,.0f}: " + ", ".join(dict.fromkeys(best.family))
        winners.append(row)
        frontier.append(distinct(pareto(practical)).sort_values("qubits_total").assign(graph=g))
    winners = pd.DataFrame(winners).set_index("graph")
    frontier = pd.concat(frontier, ignore_index=True)
    display(Markdown(f"**{PARETO_REGIME}: best value of each metric per Grover iteration, and the families that reach it** "
                     f"(configurations within {T_LIMIT}× the fewest T gates)"))
    display(winners)
    counts = frontier.assign(family=frontier.families.str.split(", ")).explode("family").pivot_table(
        index="graph", columns="family", values="configuration", aggfunc="count", fill_value=0)
    counts = counts.reindex(index=[g for g in GRAPHS if g in counts.index], columns=list(FAMILIES), fill_value=0)
    display(Markdown("**Distinct Pareto-optimal costs reached by each family** (a cost shared by two families counts for both)"))
    display(counts)
    shown = frontier[frontier.graph.isin(["K5", "K20", "myciel4", "2-Insertions_3"])]
    display(Markdown("**The frontier on four graphs** (qubits, T gates and T-depth per iteration)"))
    display(shown[["graph", "configuration", "families", *PARETO_METRICS]].astype({m: int for m in PARETO_METRICS})
            .set_index(["graph", "configuration"]))
    frontier.to_csv(OUT / f"pareto_frontier_{PARETO_REGIME}.csv", index=False)

    panels = ["K5", "K10", "myciel4", "K20"]
    figure, axes = plt.subplots(2, len(panels), figsize=(4.2 * len(panels), 7.2))
    for column, g in enumerate(panels):
        graph = part[part.benchmark == g]
        front = set(pareto(graph[graph.t_count <= T_LIMIT * graph.t_count.min()]).configuration)
        for row, (metric, label) in enumerate((("t_count", "T gates per iteration"), ("t_depth", "T-depth per iteration"))):
            ax = axes[row, column]
            for name, color in reversed(FAMILIES.items()):  # the XAG flow last, on top of ties
                points = graph[graph.family == name]
                on = points.configuration.isin(front)
                ax.scatter(points.qubits_total[~on], points[metric][~on], s=18, color=color, alpha=0.35, linewidth=0)
                ax.scatter(points.qubits_total[on], points[metric][on], s=46, color=color, edgecolor=INK, linewidth=0.9,
                           label=name if (row, column) == (0, 0) else None, zorder=4 if name == "XAG flow" else 3)
            ax.set_xscale("log")
            ax.set_yscale("log")
            log_axis(ax.xaxis)
            log_axis(ax.yaxis)
            ax.set_xlabel("Qubits")
            if column == 0:
                ax.set_ylabel(label)
            if row == 0:
                ax.set_title(g, color=INK)
    handles, names = axes[0, 0].get_legend_handles_labels()
    figure.legend(handles[::-1], names[::-1], frameon=False, ncol=4, loc="lower center", bbox_to_anchor=(0.5, -0.03))
    figure.suptitle(f"{PARETO_REGIME}: every configuration per iteration; outlined points are Pareto-optimal (within {T_LIMIT}× the fewest T gates)",
                    color=INK, y=1.0)
    figure.tight_layout(rect=(0, 0.03, 1, 0.97))
    show_figure(figure, f"pareto-{PARETO_REGIME}")
    xag_on = frontier[frontier.families.str.contains("XAG flow")].graph.nunique()
    simple_on = frontier[frontier.families.str.contains("Belletti-Simple")].graph.nunique()
    display(Markdown(
        f"The XAG flow has at least one Pareto-optimal configuration on {xag_on} of {frontier.graph.nunique()} graphs and "
        f"Belletti-Simple on {simple_on}. The frontier, not a single ratio, is the comparison the paper should report."
    ))

## 10. Two-qubit gates and routing (optional: current hardware)

*Whether the paper covers current hardware is still open; this section keeps the evidence at hand either way.*

The fault-tolerant columns price T gates, because magic states dominate the cost of an error-corrected computation.
Clifford gates are not free there either: a CNOT costs about one code cycle of lattice surgery, and as magic states
get cheaper their relative weight grows. On current (NISQ) hardware the picture changes completely: two-qubit gates
carry most of the error, every gate counts, and the device's limited connectivity forces SWAP gates whose number grows
with the circuit's width. Single-qubit Clifford gates are cheap in both settings and are not reported.

The section has three parts. First, the **CNOTs of the Clifford+T lowering** per Grover iteration, with an exchange-rate
check: if a CNOT cost α T gates, would the ranking change? Second, the **routed two-qubit gates** on the FakeTorino
snapshot (133 qubits) from the nine-graph study, where only the smallest graphs fit. Third, routing on a **square grid
that grows with the circuit** (the topology of IBM's Nighthawk processor: each qubit coupled to up to four neighbours,
CZ as the native gate), which reaches every graph and shows the effect of the conjunction shapes and gate scheduling on
routing. Routing is reported as two-qubit gate counts and two-qubit depth only.

In [ ]:
per_component = rows[rows.component.isin(["oracle", "diffusion"]) & (rows.metric_level == "fault-tolerant")]
cnots = per_component.groupby(["benchmark", "regime", "method", "mcx_policy"]).cx_count.sum(min_count=2)
iteration = per_iteration.set_index(["benchmark", "regime", "method", "mcx_policy"])
iteration = iteration.assign(cx_count=cnots.reindex(iteration.index))
PAIRS = {"B-S, throwaway": ("sb_simple", "throwaway_tree"), "B-S, Khattar–Gidney": ("sb_simple", "2_clean_kg24")}


def crossover(x, b):
    """How the comparison of cost = T + α·CNOT changes with α ≥ 0."""
    dt, dc = x.t_count - b.t_count, x.cx_count - b.cx_count
    if dt <= 0 and dc <= 0:
        return "XAG cheaper or equal for every α" if dt or dc else "equal"
    if dt >= 0 and dc >= 0:
        return "baseline cheaper for every α"
    alpha = -dt / dc
    return f"XAG cheaper while α {'<' if dc > 0 else '>'} {alpha:.1f}"


for regime in REGIMES:
    table = []
    for g in GRAPHS:
        try:
            x = iteration.loc[(g, regime, "xag", "throwaway_tree")]
        except KeyError:
            continue
        row = {"graph": g, "XAG T": int(x.t_count), "XAG CNOT": int(x.cx_count)}
        for label, (method, policy) in PAIRS.items():
            b = iteration.loc[(g, regime, method, policy)]
            row[f"{label}: T"] = int(b.t_count)
            row[f"{label}: CNOT"] = int(b.cx_count)
            row[f"vs {label}"] = crossover(x, b)
        table.append(row)
    display(Markdown(f"**{regime}: T gates and CNOTs per Grover iteration (Clifford+T lowering), and the effect of "
                     f"pricing a CNOT at α T gates**"))
    display(pd.DataFrame(table).set_index("graph"))

In [ ]:
torino = rows[(rows.metric_level == "routed") & (rows.component == "iteration")]
fits = torino[torino.routed == True]  # noqa: E712
both = fits[fits.method.isin(["xag", "sb_simple"])].groupby("benchmark").method.nunique()
display(Markdown(f"**FakeTorino (133 qubits), nine-graph study**: the XAG oracle and Belletti-Simple are both routed on "
                 f"{(both == 2).sum()} of {torino.benchmark.nunique()} graphs ({', '.join(both[both == 2].index)}); the "
                 f"others need more qubits than the device has or more than {settings['limits']['route_cx']:,} CNOTs."))
for regime in REGIMES:
    part = fits[(fits.regime == regime) & fits.method.isin(["xag", "sb_simple"])]
    if part.empty:
        continue
    best = part.loc[part.groupby(["benchmark", "method"]).two_qubit_count.idxmin()]
    table = best.pivot(index="benchmark", columns="method", values=["two_qubit_count", "two_qubit_depth", "qubits_total"])
    table = table.reindex([g for g in GRAPHS if g in table.index])
    display(Markdown(f"**{regime}**: routed iteration, XAG against Belletti-Simple (each with its decomposition of "
                     f"fewest two-qubit gates; median over five transpiler seeds)"))
    display(table.astype("Int64"))

In [ ]:
GRID_RUNS = {
    "XAG, as in the study": ("routed-grid-study", "xag"),
    **{f"XAG, {tree}, scheduled": (f"routed-grid-shapes-{tree}", "xag") for tree in TREES},
    "B-S, as in the study": ("routed-grid-study", "sb_simple"),
    "B-S, scheduled": ("routed-grid-baselines-scheduled", "sb_simple"),
}
grid_data = {}
for description in {d for d, _ in GRID_RUNS.values()}:
    run = latest_run(description)
    if run is not None:
        data = analysis.load_study(run)[0]
        grid_data[description] = data[(data.metric_level == "routed") & (data.component == "iteration")]
display(Markdown("Square-grid runs available: " + ", ".join(sorted(grid_data)) +
                 ("" if len(grid_data) == 5 else " (the others are still running; rerun this cell later)")))
grid_tables = {}
for regime in REGIMES:
    for policy, policy_name in (("throwaway_tree", "throwaway tree"), ("2_clean_kg24", "Khattar–Gidney")):
        table = pd.DataFrame(index=GRAPHS)
        for label, (description, method) in GRID_RUNS.items():
            data = grid_data.get(description)
            if data is None:
                continue
            part = data[(data.regime == regime) & (data.method == method) & (data.mcx_policy == policy) & (data.routed == True)]  # noqa: E712
            part = part.set_index("benchmark").reindex(GRAPHS)
            table[f"{label}: 2q"] = part.two_qubit_count
            table[f"{label}: 2q depth"] = part.two_qubit_depth
        if table.empty or table.isna().all().all():
            continue
        shapes = [c for c in table.columns if c.startswith("XAG, ") and "scheduled: 2q" in c and "depth" not in c]
        if shapes:
            table["XAG, best shape: 2q"] = table[shapes].min(axis=1)
            table["XAG, best shape: 2q depth"] = table[[c + " depth" for c in shapes]].min(axis=1)
        grid_tables[(regime, policy)] = table
        display(Markdown(f"**{regime}, {policy_name}**: two-qubit gates and two-qubit depth of a routed Grover "
                         f"iteration on the growing square grid (median over five seeds)"))
        display(table.dropna(how="all").astype("Int64"))
key = ("R3", "throwaway_tree")
if key in grid_tables and "XAG, best shape: 2q" in grid_tables[key] and "B-S, scheduled: 2q" in grid_tables[key]:
    t = grid_tables[key].dropna(subset=["XAG, as in the study: 2q", "XAG, best shape: 2q", "B-S, scheduled: 2q"])
    figure, axes = plt.subplots(1, 2, figsize=(12, 3.6))
    for ax, suffix, title in ((axes[0], "2q", "two-qubit gates"), (axes[1], "2q depth", "two-qubit depth")):
        x = np.arange(len(t))
        for k, (label, color) in enumerate(zip(["XAG, as in the study", "XAG, best shape", "B-S, as in the study", "B-S, scheduled"],
                                                [SERIES[1], SERIES[0], SERIES[4], SERIES[3]])):
            column = f"{label}: {suffix}"
            if column in t:
                ax.bar(x + (k - 1.5) * 0.2, t[column], width=0.2, color=color, edgecolor="white", label=label)
        ax.set_yscale("log")
        log_axis(ax.yaxis)
        ax.set_xticks(x, t.index, rotation=30, ha="right")
        ax.set_title(f"R3, throwaway tree: {title} per routed iteration", color=INK)
        ax.grid(axis="x", visible=False)
    axes[0].legend(frameon=False, fontsize=8)
    show_figure(figure, "routing-grid")

In [ ]:
# What scheduling and the conjunction shapes change after routing, and the XAG flow against the best
# hand-constructed approach for each metric (all four Saha–Belletti constructions, scheduled, either decomposition).
def geomean(values):
    values = values.dropna()
    return float(np.exp(np.log(values).mean())) if len(values) else np.nan


KEYS = ["benchmark", "regime", "method", "mcx_policy"]
ROUTED_METRICS = {"two_qubit_count": "2q", "two_qubit_depth": "2q depth"}
plain, scheduled = grid_data["routed-grid-study"], grid_data["routed-grid-baselines-scheduled"]
plain, scheduled = plain[plain.routed == True], scheduled[scheduled.routed == True]  # noqa: E712
grid_limit = json.loads((latest_run("routed-grid-study") / "run.json").read_text())["settings"]["limits"]["route_cx"]
skipped = grid_data["routed-grid-study"].query("routed == False").groupby(["benchmark", "method"]).size()
display(Markdown(
    f"Every XAG case was routed. {int(skipped.sum())} hand-constructed cases exceed the routing limit of "
    f"{grid_limit:,} CNOTs and are left out: "
    + ", ".join(f"{NAMES.get(m, m)} on {g} ({n})" for (g, m), n in skipped.items()) + "."))

# 1. Scheduling the hand-constructed approaches.
pairs = plain.merge(scheduled, on=KEYS, suffixes=("", " scheduled"))
effect = []
for (method, policy), part in pairs[pairs.method != "xag"].groupby(["method", "mcx_policy"]):
    row = {"approach": NAMES.get(method, method), "decomposition": POLICY_NAMES.get(policy, policy)}
    for metric, label in ROUTED_METRICS.items():
        ratio = part[f"{metric} scheduled"] / part[metric]
        row[f"{label}: geometric mean"] = round(geomean(ratio), 3)
        row[f"{label}: range"] = f"{ratio.min():.2f}–{ratio.max():.2f}"
    effect.append(row)
display(Markdown("**Scheduling the hand-constructed approaches**: routed iteration after scheduling, relative to the "
                 "same circuit unscheduled (all regimes and graphs)"))
display(pd.DataFrame(effect).set_index(["approach", "decomposition"]))

# 2. Shapes, balancing and scheduling of the XAG flow.
xag_plain = plain[plain.method == "xag"]
shape_rows = []
for tree in TREES:
    data = grid_data.get(f"routed-grid-shapes-{tree}")
    merged = xag_plain.merge(data, on=KEYS, suffixes=("", " shaped"))
    row = {"conjunction shape (balanced, scheduled)": tree}
    for metric, label in ROUTED_METRICS.items():
        ratio = merged[f"{metric} shaped"] / merged[metric]
        row[f"{label}: geometric mean"] = round(geomean(ratio), 3)
        row[f"{label}: range"] = f"{ratio.min():.2f}–{ratio.max():.2f}"
    shape_rows.append(row)
display(Markdown("**The XAG flow with each conjunction shape, balanced and scheduled**, relative to the XAG flow as in "
                 "the study (unbalanced, unscheduled); the qubits are the same"))
display(pd.DataFrame(shape_rows).set_index("conjunction shape (balanced, scheduled)"))

# 3. Against the best hand-constructed approach for each metric.
shaped = pd.concat([grid_data[f"routed-grid-shapes-{tree}"] for tree in TREES])
verdicts = []
for regime in REGIMES:
    table = pd.DataFrame(index=GRAPHS)
    for metric, label in ROUTED_METRICS.items():
        ours = shaped[shaped.regime == regime].groupby("benchmark")[metric].min()
        theirs = scheduled[(scheduled.regime == regime) & (scheduled.method != "xag")]
        best = theirs.loc[theirs.groupby("benchmark")[metric].idxmin()].set_index("benchmark")
        ratio = (ours / best[metric]).reindex(GRAPHS)
        table[f"{label}: XAG / best"] = ratio.round(2)
        table[f"{label}: best hand-constructed"] = (
            best.method.map(NAMES) + ", " + best.mcx_policy.map(POLICY_NAMES)).reindex(GRAPHS)
        verdicts.append({"regime": regime, "metric": label, "geometric mean": round(geomean(ratio), 2),
                         "wins / ties / losses": f"{(ratio < 0.995).sum()} / {ratio.between(0.995, 1.005).sum()} / {(ratio > 1.005).sum()}"})
    display(Markdown(f"**{regime}**: the XAG flow (best of the three shapes, balanced and scheduled, either "
                     f"decomposition) against the best hand-constructed approach for each metric, routed on the grid"))
    display(table)
display(Markdown("**Summary across the nine graphs**"))
display(pd.DataFrame(verdicts).set_index(["regime", "metric"]))

**Reading the routed results.**

* *Scheduling barely changes the routed cost of the hand-constructed approaches.* Their geometric means move by at most
  a few percent in either metric, with single cases between about −18% and +13% in depth. The scheduler reorders
  commuting gates to shorten the T-depth, but the router then inserts SWAP gates and reorders gates again, so what
  scheduling gains in the logical circuit is mostly lost after routing.
* *The conjunction shapes, with balancing and scheduling, keep their effect after routing for the XAG flow.* They lower
  the routed two-qubit depth by about a fifth to a quarter (geometric mean) at nearly the same gate count, because they
  change which constraints are combined, not only the order of the gates. On the largest graphs (K₂₀, and myciel5 with
  the color-range checks) the shapes that pair distant constraints need noticeably more SWAP gates than the input
  order, which keeps each vertex's constraints together; a shape chosen for a device would have to model distance on
  the device as well as readiness.
* *Against the best hand-constructed approach for each metric*, the XAG flow routes to fewer two-qubit gates and a
  lower two-qubit depth on most graphs when the color-range checks are kept (R1, R2), with the largest margins on K₁₀
  and K₂₀, where the synthesized color-range checks are cheapest relative to the hand-built ones. Without the checks
  (R3), where both compute the same AND gates, the XAG flow needs somewhat more two-qubit gates on most graphs (its
  extra AND workspace widens the grid) and ties in depth. The best hand-constructed approach for the gate count is
  mostly Belletti-Simple with Khattar–Gidney (narrowest), and for depth Belletti-Simple with the throwaway tree.
* On the graphs with four colors (myciel3, 2-Insertions_3, zed-city_2019) there are no unused codes, so the three
  regimes produce the same circuit and the same routed figures.
* These figures are optional material: whether the paper covers current hardware is decided later.

In [ ]:
from logique.benchmarks.provenance import checkout_revision, environment

overview.to_csv(OUT / "graphs.csv")
widths.to_csv(OUT / "width_accounting.csv")
summary.to_csv(OUT / "comparison_summary.csv", index=False)
equivalence.to_csv(OUT / "equivalence.csv")
choices.to_csv(OUT / "preparations.csv", index=False)
record = {
    "study": manifest["run_id"],
    "key_results": {k: (v if not isinstance(v, dict) else {str(a): b for a, b in v.items()}) for k, v in KEY.items()},
    "versions": environment(),
    "checkout": checkout_revision(ROOT),
}
(OUT / "overview_manifest.json").write_text(json.dumps(record, indent=2, default=str) + "\n")
print("Saved:", sorted(p.name for p in OUT.iterdir()))